# Data

In this notebook I go through everything I did with the data before modelling: checking I have the right file, the audit, cleaning, and building the inputs for my models.

In [1]:
# Add the project folder to the path so `from src import ...` works when this
# notebook is run from the notebooks folder.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.audit import check_snapshot, run
from src.clean import clean
from src.features import FEATURES, build_features, check_no_leakage
from src.paths import INTERIM, RAW_ZIP

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)

Python: 3.13.15 | pandas: 3.0.5


## 1. Question and data

I want to predict whether a powerlifter's third deadlift will be successful. I only use information that's known once they've picked the weight for that attempt, so nothing from during or after the lift.

I'm using the OpenPowerlifting IPF data, snapshot `openipf-2026-10-03`, which I downloaded on 4 October 2026. It's public domain. The download link always gives the newest version, so the cell below checks my ZIP is exactly the file I used.

In [2]:
check_snapshot(RAW_ZIP)

'openipf-2026-10-03/openipf-2026-10-03-199bb416.csv'

## 2. Data audit

Here I filter the data down to the lifts I can actually use and remove duplicates. I end up with 462,489 lifts, and about 61% of third attempts are successful. My full write-up of the audit is in `reports/data_audit.md`.

In [3]:
results = run()
counts, final = results['counts'], results['final']

filter_steps = ['raw_rows', 'SBD', 'SBD_Raw', 'three_attempts_present', 'valid_bodyweight',
                'valid_nonzero_attempts', 'final_unique_recorded_performances']
pd.DataFrame({'rows left': [counts[k] for k in filter_steps]}, index=filter_steps)

,rows left
raw_rows,1529372
SBD,1135955
SBD_Raw,624937
three_attempts_present,481386
valid_bodyweight,481248
valid_nonzero_attempts,481248
final_unique_recorded_performances,462489


In [4]:
split = pd.DataFrame({'rows': [final['success'], final['failure']]}, index=['success', 'failure'])
split['pct'] = (100 * split.rows / split.rows.sum()).round(2)
split

,rows,pct
success,281925,60.96
failure,180564,39.04


In [5]:
flags = ['age_below_10', 'third_attempt_above_450kg', 'bodyweight_below_30_or_above_250',
         'repeated_identity_meet_rows', 'repeated_identity_meet_groups',
         'third_attempt_same_load_as_second']
pd.Series({k: counts[k] for k in flags}, name='rows')

age_below_10                           344
third_attempt_above_450kg                2
bodyweight_below_30_or_above_250       113
repeated_identity_meet_rows            659
repeated_identity_meet_groups          329
third_attempt_same_load_as_second    38058
Name: rows, dtype: int64

## 3. Cleaning

Next I apply my cleaning rules. I remove the 659 rows where the same person shows up more than once at the same meet, because I can't tell which entry is right, and the 2 third attempts over 450 kg, which are probably data-entry errors. I keep the very young and very light lifters because they're real. My reasons for each rule are in `reports/cleaning_decisions.md`.

In [6]:
cohort = pd.read_csv(INTERIM / 'audit_cohort.csv.gz', dtype=str, keep_default_na=False)
cleaned, clean_counts = clean(cohort)
pd.Series(clean_counts, name='rows')

start_rows                               462489
removed_repeated_identity_meet              659
removed_third_attempt_above_450kg             2
kept_age_below_10                           344
kept_bodyweight_below_30_or_above_250       113
final_rows                               461828
Name: rows, dtype: int64

## 4. Inputs

Finally I turn the cleaned rows into the table I'll train my models on. `made` is what I'm predicting (1 if the third deadlift was successful). I only keep `Name` so each lifter stays on one side of the train/test split, so I never show it or use it as an input. The last cell checks that none of the columns that would give the answer away have snuck in.

In [7]:
table = build_features(cleaned)
table.drop(columns='Name').head()

,Sex,Age,BodyweightKg,deadlift1_kg,deadlift2_kg,deadlift3_kg,deadlift1_made,deadlift2_made,jump_kg,jump_pct,load_per_bodyweight,made,Date
0,F,41.0,107.5,145.0,157.5,162.5,1,1,5.0,3.174603,1.511628,0,2025-08-17
1,F,36.0,56.3,125.0,135.0,142.5,1,1,7.5,5.555556,2.531083,1,2025-08-17
2,F,24.0,67.5,140.0,152.5,162.5,1,1,10.0,6.557377,2.407407,1,2025-08-17
3,F,39.0,74.8,117.5,127.5,132.5,1,1,5.0,3.921569,1.771390,1,2025-08-17
4,F,25.0,82.3,170.0,185.0,192.5,1,1,7.5,4.054054,2.339004,0,2025-08-17


In [8]:
table.drop(columns='Name').dtypes

Sex                        str
Age                    float64
BodyweightKg           float64
deadlift1_kg           float64
deadlift2_kg           float64
deadlift3_kg           float64
deadlift1_made           int64
deadlift2_made           int64
jump_kg                float64
jump_pct               float64
load_per_bodyweight    float64
made                     int64
Date                       str
dtype: object

In [9]:
table.drop(columns='Name').isna().sum()

Sex                        0
Age                    73373
BodyweightKg               0
deadlift1_kg               0
deadlift2_kg               0
deadlift3_kg               0
deadlift1_made             0
deadlift2_made             0
jump_kg                    0
jump_pct                   0
load_per_bodyweight        0
made                       0
Date                       0
dtype: int64

In [10]:
check_no_leakage(table[FEATURES])
print(f'No leakage: the {len(FEATURES)} input columns are all known before the third attempt.')

No leakage: the 11 input columns are all known before the third attempt.
